In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

### data

In [2]:
n: int = 40_000
rng = np.random.default_rng(0)
a = rng.choice(["seg_low", "seg_mid", "seg_high"], size=n, p=[.5, .3, .2])
b = rng.integers(0, 4, size=n)                    # ordinal 0..3, REAL signal
df = pd.DataFrame({
    "a": a,                                       # 3 levels, real signal
    "b": b,                                       # 4 levels, real signal
    "c": rng.choice(list("WXYZ"), size=n),        # noise, 4 levels
    "d": rng.choice([f"d{i}" for i in range(5)],  # noise, 5 levels
                    size=n),
    "e": rng.integers(0, 5, size=n),              # noise, 5 levels (ints)
    "f": rng.choice(["p", "q"], size=n),          # noise, 2 levels
    "g": rng.choice(["g_lo", "g_mid", "g_hi"],    # noise, 3 levels
                    size=n),
    # 10 discrete price points: 1.0, 2.0, ..., 10.0
    "bid": rng.choice(np.arange(1.0, 11.0), size=n),
})
threshold = df["a"].map({"seg_low": 2.5, "seg_mid": 4.5,
                            "seg_high": 6.5}).to_numpy()
p_win = 1.0 / (1.0 + np.exp(-(df["bid"].to_numpy()
                                - threshold - 0.7 * b) / 0.9))
df["bid_won"] = rng.binomial(1, p_win)

print(df.shape)
df.head(22)

(40000, 9)


,a,b,c,d,e,f,g,bid,bid_won
0,seg_mid,2,X,d4,1,q,g_mid,8.0,1
1,seg_low,0,Y,d0,3,p,g_lo,4.0,1
2,seg_low,1,X,d3,3,q,g_hi,6.0,1
3,seg_low,1,W,d3,1,p,g_hi,4.0,1
4,seg_high,1,Z,d2,4,q,g_mid,2.0,0
5,seg_high,1,Z,d1,2,q,g_lo,6.0,0
6,seg_mid,3,X,d4,3,p,g_hi,1.0,0
7,seg_mid,0,Z,d2,3,q,g_hi,4.0,0
8,seg_mid,2,W,d0,4,q,g_hi,8.0,1
9,seg_high,0,Y,d1,4,p,g_hi,3.0,0


In [4]:
candidate_features: Tuple[str, ...] = ("a", "b", "c", "d", "e", "f", "g")
bid_col: str = "bid"
target_col: str = "bid_won"
max_expected_cardinality: int = 50

In [5]:
for col in list(candidate_features) + [bid_col]:
    k = df[col].nunique()
    if k > max_expected_cardinality:
        raise ValueError(
            f"Column '{col}' has {k} distinct values, which exceeds "
            f"max_expected_cardinality={max_expected_cardinality}. "
            f"This module assumes pre-discretized features and a small "
            f"set of bid price points; discretize '{col}' upstream or "
            f"raise the limit if this is intentional."
        )

In [6]:
feat_str = pd.DataFrame(index=df.index)
for col in candidate_features:
    feat_str[col] = df[col].astype(str)
print(feat_str.shape)
feat_str

(40000, 7)


,a,b,c,d,e,f,g
0,seg_mid,2,X,d4,1,q,g_mid
1,seg_low,0,Y,d0,3,p,g_lo
2,seg_low,1,X,d3,3,q,g_hi
3,seg_low,1,W,d3,1,p,g_hi
4,seg_high,1,Z,d2,4,q,g_mid
...,...,...,...,...,...,...,...
39995,seg_mid,2,W,d3,3,p,g_hi
39996,seg_mid,2,Y,d1,4,q,g_lo
39997,seg_low,0,X,d2,4,q,g_hi
39998,seg_mid,0,W,d2,2,p,g_lo


In [7]:
levels = np.sort(df['bid'].unique())
bid_levels = pd.Series(pd.Categorical(df['bid'], categories=levels, ordered=True),index=df['bid'].index)
bid_levels


0        8.0
1        4.0
2        6.0
3        4.0
4        2.0
        ... 
39995    2.0
39996    4.0
39997    2.0
39998    8.0
39999    4.0
Length: 40000, dtype: category
Categories (10, float64): [1.0 < 2.0 < 3.0 < 4.0 ... 7.0 < 8.0 < 9.0 < 10.0]

In [8]:
y = df['bid_won']
max_size = len(candidate_features)

In [12]:
def wilson_interval(wins: np.ndarray, n: np.ndarray, alpha: float = 0.05
                    ) -> Tuple[np.ndarray, np.ndarray]:
    wins = np.asarray(wins, dtype=float)
    n = np.asarray(n, dtype=float)
    z = stats.norm.ppf(1.0 - alpha / 2.0)          # e.g. 1.96 for alpha=0.05

    with np.errstate(divide="ignore", invalid="ignore"):
        p_hat = np.where(n > 0, wins / n, np.nan)  # empirical win rate
        denom = 1.0 + z**2 / n
        centre = (p_hat + z**2 / (2.0 * n)) / denom
        half = (z * np.sqrt(p_hat * (1.0 - p_hat) / n
                            + z**2 / (4.0 * n**2))) / denom

    lo = np.clip(centre - half, 0.0, 1.0)
    hi = np.clip(centre + half, 0.0, 1.0)
    # Empty cells: no information at all -> [0, 1], i.e. width 1.
    lo = np.where(n > 0, lo, 0.0)
    hi = np.where(n > 0, hi, 1.0)
    return lo, hi


def clopper_pearson_interval(wins: np.ndarray, n: np.ndarray,
                             alpha: float = 0.05
                             ) -> Tuple[np.ndarray, np.ndarray]:
    wins = np.asarray(wins, dtype=float)
    n = np.asarray(n, dtype=float)

    with np.errstate(invalid="ignore"):
        # Standard CP construction; the np.where guards handle the edge cases
        # k == 0 (lower bound is exactly 0) and k == n (upper bound exactly 1),
        # where the Beta quantile would be undefined.
        lo = np.where(wins > 0,
                      stats.beta.ppf(alpha / 2.0, wins, n - wins + 1.0), 0.0)
        hi = np.where(wins < n,
                      stats.beta.ppf(1.0 - alpha / 2.0, wins + 1.0, n - wins),
                      1.0)

    lo = np.where(n > 0, np.nan_to_num(lo, nan=0.0), 0.0)
    hi = np.where(n > 0, np.nan_to_num(hi, nan=1.0), 1.0)
    return lo, hi


def _interval(wins, n):
    if True:
        return wilson_interval(wins, n, alpha=0.05)
    if cfg.ci_method == "clopper_pearson":
        return clopper_pearson_interval(wins, n, alpha=0.05)
    raise ValueError(f"Unknown ci_method: {cfg.ci_method!r}")

def make_context_key(feat_str: pd.DataFrame,
                     subset: Tuple[str, ...]) -> pd.Series:
    if len(subset) == 0:
        return pd.Series("GLOBAL", index=feat_str.index)
    return feat_str[list(subset)].agg("|".join, axis=1)

@dataclass
class SufficiencyReport:
    subset: Tuple[str, ...]
    passes: bool                    # median width <= cfg.max_median_ci_width ?
    median_ci_width: float          # THE rejection statistic
    traffic_weighted_width: float   # extra diagnostic (weights = cell size)
    n_contexts: int                 # how many contexts the subset induces
    n_cells: int                    # contexts x bid levels actually scored
    frac_empty_cells: float         # coverage gaps in the full grid
    cell_table: pd.DataFrame        # per-cell n / wins / rate / CI


def sufficiency_check(y: pd.Series, ctx: pd.Series, bid_level: pd.Series, subset: Tuple[str, ...]) -> SufficiencyReport:
    cells = pd.DataFrame({
        "ctx": ctx.to_numpy(),
        "bid": bid_level.to_numpy(),       # raw price values; groupable
        "y": y.to_numpy(),
    })

    # Per-cell counts: n = auctions in the cell, wins = won auctions.
    agg = (cells.groupby(["ctx", "bid"], observed=True)["y"]
                .agg(n="size", wins="sum"))

    full_grid = pd.MultiIndex.from_product(
        [np.unique(cells["ctx"]), list(bid_level.cat.categories)],
        names=["ctx", "bid"],
    )
    agg = agg.reindex(full_grid, fill_value=0)

    n = agg["n"].to_numpy(dtype=float)
    wins = agg["wins"].to_numpy(dtype=float)

    lo, hi = _interval(wins, n)
    width = hi - lo                                  # empty cells -> 1.0

    median_width = float(np.median(width))
    # Traffic-weighted mean width: "how uncertain is the curve for the
    # average auction?" (empty cells naturally drop out, weight 0).
    populated = n > 0
    weighted_width = (float(np.average(width[populated], weights=n[populated]))
                      if populated.any() else 1.0)

    table = agg.reset_index()
    table["win_rate"] = np.where(n > 0, wins / np.maximum(n, 1), np.nan)
    table["ci_lo"], table["ci_hi"], table["ci_width"] = lo, hi, width

    return SufficiencyReport(
        subset=subset,
        passes=median_width <= 0.30,
        median_ci_width=median_width,
        traffic_weighted_width=weighted_width,
        n_contexts=int(cells["ctx"].nunique()),
        n_cells=int(len(agg)),
        frac_empty_cells=float(np.mean(n == 0)),
        cell_table=table,
    )

In [15]:
def _fit_isotonic(bids: np.ndarray, y: np.ndarray) -> IsotonicRegression:
    return IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True,
                              out_of_bounds="clip").fit(bids, y)


@dataclass
class CVResult:
    subset: Tuple[str, ...]
    mean_log_loss: float
    std_log_loss: float
    fold_log_losses: List[float]
    # Share of validation rows scored by their OWN context model (the rest
    # fell back to the global curve).  Low values reveal fragmentation even
    # before the sufficiency gate does.
    frac_context_scored: float


def cv_log_loss_for_subset(df: pd.DataFrame, feat_str: pd.DataFrame, subset: Tuple[str, ...]) -> CVResult:
    y_all = df['bid_won'].to_numpy()
    bids = df['bid'].to_numpy(dtype=float)
    ctx = make_context_key(feat_str, subset).to_numpy()

    # NOTE (temporal data): bidding logs usually drift over time.  For a
    # production system replace StratifiedKFold with time-ordered splits,
    # e.g. sklearn.model_selection.TimeSeriesSplit, keeping the rest as-is.
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    losses: List[float] = []
    ctx_scored: List[float] = []

    for train_idx, val_idx in skf.split(np.zeros(len(df)), y_all):
        # ---- fit: one global fallback + one model per big-enough context ----
        global_model = _fit_isotonic(bids[train_idx], y_all[train_idx])

        train_frame = pd.DataFrame({"ctx": ctx[train_idx],
                                    "bid": bids[train_idx],
                                    "y": y_all[train_idx]})
        models: Dict[str, IsotonicRegression] = {}
        for c, g in train_frame.groupby("ctx", sort=False):
            # A context earns its own curve only with enough rows AND at
            # least two distinct bid values (otherwise no curve to speak of).
            # Contexts with constant y still get a curve (a constant one) --
            # if that constant is overconfident, CV log-loss will punish it,
            # which is precisely the selection mechanism at work.
            if len(g) >= 200 and g["bid"].nunique() >= 2:
                models[c] = _fit_isotonic(g["bid"].to_numpy(),
                                          g["y"].to_numpy())

        # ---- predict on the validation fold, context by context -------------
        preds = np.empty(len(val_idx), dtype=float)
        val_frame = pd.DataFrame({"ctx": ctx[val_idx], "bid": bids[val_idx]},
                                 index=np.arange(len(val_idx)))
        n_context_scored = 0
        for c, g in val_frame.groupby("ctx", sort=False):
            model = models.get(c)
            if model is None:                 # small or unseen context
                model = global_model
            else:
                n_context_scored += len(g)
            preds[g.index] = model.predict(g["bid"].to_numpy())

        # ---- score -----------------------------------------------------------
        preds = np.clip(preds, 1e-6, 1.0 - 1e-6)
        losses.append(log_loss(y_all[val_idx], preds, labels=[0, 1]))
        ctx_scored.append(n_context_scored / len(val_idx))

    return CVResult(subset=subset,
                    mean_log_loss=float(np.mean(losses)),
                    std_log_loss=float(np.std(losses)),
                    fold_log_losses=[float(l) for l in losses],
                    frac_context_scored=float(np.mean(ctx_scored)))


### step 0: baseline = no contextualization

In [16]:
base_subset: Tuple[str, ...] = ()
base_suff = sufficiency_check(y, make_context_key(feat_str, base_subset),bid_levels, base_subset)
base_cv = cv_log_loss_for_subset(df, feat_str, base_subset)

### greedy loop

In [19]:
def _fmt_subset(subset: Tuple[str, ...]) -> str:
    return "{" + ", ".join(subset) + "}" if subset else "{} (global)"

best_subset, best_loss = base_subset, base_cv.mean_log_loss
remaining = list(candidate_features)
step = 0

In [20]:
while remaining and len(best_subset) < max_size:
    step += 1
    passing: List[Tuple[CVResult, SufficiencyReport]] = []

    for feat in remaining:
        candidate = best_subset + (feat,)
        ctx = make_context_key(feat_str, candidate)

        # (1) sufficiency gate -- reject before paying for CV.
        suff = sufficiency_check(y, ctx, bid_levels, candidate)
        if not suff.passes:
            print(f"[step {step}] REJECT {_fmt_subset(candidate)} : "
                f"median CI width {suff.median_ci_width:.3f} "
                f"> {0.30} "
                f"({suff.n_contexts} contexts, "
                f"{suff.frac_empty_cells:.0%} empty cells)")
            continue

        # (2) accuracy -- CV log-loss of the per-context isotonic model.
        cv = cv_log_loss_for_subset(df, feat_str, candidate)
        passing.append((cv, suff))
        print(f"[step {step}] eval   {_fmt_subset(candidate)} : "
                f"CV log-loss = {cv.mean_log_loss:.5f} "
                f"(+/-{cv.std_log_loss:.5f}), "
                f"median CI width = {suff.median_ci_width:.3f}, "
                f"context-scored rows = {cv.frac_context_scored:.0%}")

    if not passing:
        break

    cv_best, suff_best = min(passing, key=lambda t: t[0].mean_log_loss)
    improvement = best_loss - cv_best.mean_log_loss

    if improvement > 0.002:
        best_subset = cv_best.subset
        best_loss = cv_best.mean_log_loss
        remaining.remove(best_subset[-1])         # consume the feature
        print(f"[step {step}] ACCEPT {_fmt_subset(best_subset)} (improvement {improvement:.5f})")
    else:
        print(f"[step {step}] best improvement {improvement:.5f} <= tolerance 0.002 -> stop.")
        break



[step 1] eval   {a} : CV log-loss = 0.31400 (+/-0.00486), median CI width = 0.030, context-scored rows = 100%
[step 1] eval   {b} : CV log-loss = 0.39360 (+/-0.00691), median CI width = 0.043, context-scored rows = 100%
[step 1] eval   {c} : CV log-loss = 0.41585 (+/-0.00640), median CI width = 0.044, context-scored rows = 100%
[step 1] eval   {d} : CV log-loss = 0.41595 (+/-0.00632), median CI width = 0.051, context-scored rows = 100%
[step 1] eval   {e} : CV log-loss = 0.41593 (+/-0.00615), median CI width = 0.049, context-scored rows = 100%
[step 1] eval   {f} : CV log-loss = 0.41539 (+/-0.00642), median CI width = 0.031, context-scored rows = 100%
[step 1] eval   {g} : CV log-loss = 0.41545 (+/-0.00636), median CI width = 0.038, context-scored rows = 100%
[step 1] ACCEPT {a} (improvement 0.10128)
[step 2] eval   {a, b} : CV log-loss = 0.28619 (+/-0.00503), median CI width = 0.052, context-scored rows = 100%
[step 2] eval   {a, c} : CV log-loss = 0.31729 (+/-0.00417), median CI widt